In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_excel("loan_eligibility_dataset.xlsx")

In [ ]:
df

In [ ]:
df["Dependents"] = df["Dependents"].replace("3+", "3")
df

In [ ]:

df.drop("Applicant_ID", axis=1)


for column in df.columns:

    if df[column].dtype in ["object", "string"]:
        df[column] = df[column].fillna(df[column].mode()[0])
    else:
        df[column] = df[column].fillna(df[column].median())


encoders = {}


for column in df.select_dtypes(include=['string','object']).columns:
    lb = LabelEncoder()
    df[column] = lb.fit_transform(df[column])
    encoders[column] = lb
df

In [ ]:
x = df.drop(["Applicant_ID", "Loan_Status"], axis=1)
x

In [ ]:
y = df[["Loan_Status"]]
y

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestClassifier(criterion='entropy', max_depth=3)
model.fit(xtrain, ytrain)

In [ ]:
ypred = model.predict(xtest)
ypred

In [ ]:
print("Accuracy : ", accuracy_score(ytest, ypred)*100)

In [ ]:
model = RandomForestClassifier(criterion='entropy', max_depth=3)
model.fit(xtrain, ytrain)

In [ ]:
ypred = model.predict(xtest)
ypred

In [ ]:
print("Accuracy : ", accuracy_score(ytest, ypred)*100)

In [ ]:
for i, tree in enumerate(model.estimators_):

    plt.figure(figsize=(20, 10))

    plot_tree(tree, feature_names=x.columns, class_names=["Rejected", "Accepted"], filled=True)

    plt.title(f"Random Tree Classifier{i+1}")

    plt.show()

In [ ]:
new_user = pd.DataFrame([{
    "Gender":"Male",
    "Married":"Yes",
    "Dependents":"0",
    "Education":"Graduate",
    "Self_Employed":"No",
    "Applicant_Income":5000,
    "Coapplicant_Income":1500,
    "Loan_Amount":8000000,
    "Credit_History":1,
    "Property_Area":"Urban"

}])
print(x.columns.tolist())
print(new_user.columns.tolist())

for column, encoder in encoders.items():
    if column in new_user.columns:
        new_user[column]=encoder.transform(new_user[column].astype(str))

new_user = new_user[x.columns]

prediction = model.predict(new_user)

loan_status = encoders["Loan_Status"].inverse_transform(prediction)

print("loan_eligibility:", loan_status[0])